In [1]:
%pip install pandas numpy scikit-learn statsmodels patsy torch openpyxl matplotlib tqdm


  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 2.8 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/9.8 MB 2.8 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.8 MB 3.5 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/9.8 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.8 MB 3.9 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.8 MB 4.5 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 4.9 MB/s eta 0:00:01
   ------

### How to run
Run the setup cell once, then run the experiment cell below. The previous `python run_cann_experiment.py` command was removed because this notebook already contains the full experiment code and no such script exists in the folder.


In [3]:
"""
GLM vs DNN vs CANN experiment on cleaned MTPL data.

Data path:
C:\\Users\\Guorui\\Desktop\\4TH ULM\\SEMINAR-INSURANCE\\MTPL_data_clean.csv

Models:
1. Poisson GLM
2. Ordinary DNN
3. CANN = GLM baseline + neural network residual correction

Target:
N_i ~ Poisson(expo_i * lambda(x_i))

Important:
- truefreq is NOT used as model input.
- expo is NOT used as neural network feature.
- expo is used only as exposure / offset.
"""

import os
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. Configuration
# ============================================================

DATA_PATH = r"C:\Users\Guorui\Desktop\4TH ULM\SEMINAR-INSURANCE\MTPL_data_clean.csv"
RESULTS_DIR = r"C:\Users\Guorui\Desktop\4TH ULM\SEMINAR-INSURANCE\results"

SMOKE_TEST = True
SMOKE_N_ROWS = 10000

RANDOM_STATE = 12345

if SMOKE_TEST:
    SEEDS = [1]
    MAX_EPOCHS = 3
else:
    SEEDS = [1, 2, 3]
    MAX_EPOCHS = 100

BATCH_SIZE = 8192
PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPS = 1e-12


# ============================================================
# 2. Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_results_dir() -> None:
    os.makedirs(RESULTS_DIR, exist_ok=True)


# ============================================================
# 3. Data loading and preprocessing
# ============================================================

def load_raw_data(path: str) -> pd.DataFrame:
    """
    Load cleaned CSV data.

    sep=None allows pandas to infer whether the CSV uses comma or semicolon.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}")

    df = pd.read_csv(path, sep=None, engine="python")

    required_columns = [
        "id", "claims", "expo", "age", "ac", "power",
        "gas", "brand", "area", "dens", "ct", "truefreq"
    ]

    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}\n"
            f"Existing columns are: {list(df.columns)}"
        )

    df = df[required_columns].copy()

    numeric_cols = ["id", "claims", "expo", "age", "ac", "power", "dens", "truefreq"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="raise")

    categorical_cols = ["gas", "brand", "area", "ct"]
    for col in categorical_cols:
        df[col] = df[col].astype(str)

    return df


def add_model_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create GLM-style engineered features.
    """
    df = df.copy()

    df["log_dens"] = np.log(np.clip(df["dens"].astype(float), EPS, None))

    area_mapping = {
        "A": 1,
        "B": 2,
        "C": 3,
        "D": 4,
        "E": 5,
        "F": 6,
    }

    df["area_num"] = df["area"].map(area_mapping)

    if df["area_num"].isna().any():
        bad_values = df.loc[df["area_num"].isna(), "area"].unique()
        raise ValueError(f"Unmapped area values: {bad_values}")

    age_bins = [18, 21, 26, 31, 41, 51, 61, 71, 91]
    age_labels = [
        "18to20", "21to25", "26to30", "31to40",
        "41to50", "51to60", "61to70", "71to90"
    ]

    df["ageGLM"] = pd.cut(
        df["age"],
        bins=age_bins,
        labels=age_labels,
        right=False,
        include_lowest=True,
    )

    if df["ageGLM"].isna().any():
        bad_ages = df.loc[df["ageGLM"].isna(), "age"].unique()
        raise ValueError(f"Some ages fall outside GLM bins: {bad_ages}")

    df["ageGLM"] = df["ageGLM"].astype(str)

    df["acGLM"] = np.minimum(df["ac"].astype(int), 3)
    df["acGLM"] = df["acGLM"].map({
        0: "ac0",
        1: "ac1",
        2: "ac2",
        3: "ac3plus",
    })

    df["powerGLM"] = np.minimum(df["power"].astype(int), 9).astype(str)

    return df


def make_train_val_test_split(df: pd.DataFrame):
    """
    train: 60%
    validation: 20%
    test: 20%

    Stratified by claims > 0.
    """
    claim_indicator = (df["claims"] > 0).astype(int)

    train_df, temp_df = train_test_split(
        df,
        test_size=0.40,
        random_state=RANDOM_STATE,
        stratify=claim_indicator,
    )

    temp_indicator = (temp_df["claims"] > 0).astype(int)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp_indicator,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def make_one_hot_encoder():
    """
    Compatible with different sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_nn_preprocessor(train_df: pd.DataFrame) -> ColumnTransformer:
    """
    NN input features.

    Excluded:
    - claims
    - expo
    - id
    - truefreq
    """
    continuous_features = ["age", "ac", "power", "log_dens"]
    categorical_features = ["gas", "brand", "area", "ct"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("continuous", StandardScaler(), continuous_features),
            ("categorical", make_one_hot_encoder(), categorical_features),
        ],
        remainder="drop",
    )

    preprocessor.fit(train_df)
    return preprocessor


def transform_nn_features(preprocessor: ColumnTransformer, df: pd.DataFrame) -> np.ndarray:
    X = preprocessor.transform(df)

    if hasattr(X, "toarray"):
        X = X.toarray()

    return X.astype(np.float32)


# ============================================================
# 4. Metrics
# ============================================================

def poisson_deviance(y_true, mu_pred) -> float:
    """
    Mean Poisson deviance on claim count scale.

    y_true = observed claims N_i
    mu_pred = predicted expected claim count expo_i * lambda_hat_i
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    mu_pred = np.asarray(mu_pred, dtype=np.float64)
    mu_pred = np.clip(mu_pred, EPS, None)

    dev = np.zeros_like(y_true, dtype=np.float64)

    zero_mask = y_true == 0
    positive_mask = y_true > 0

    dev[zero_mask] = 2.0 * mu_pred[zero_mask]

    dev[positive_mask] = 2.0 * (
        mu_pred[positive_mask]
        - y_true[positive_mask]
        - y_true[positive_mask] * np.log(mu_pred[positive_mask] / y_true[positive_mask])
    )

    return float(np.mean(dev))


def estimation_loss_truefreq(expo, lambda_pred, truefreq) -> float:
    """
    Synthetic-data-only evaluation metric.

    truefreq is never used for training.
    """
    expo = np.asarray(expo, dtype=np.float64)
    lambda_pred = np.clip(np.asarray(lambda_pred, dtype=np.float64), EPS, None)
    truefreq = np.clip(np.asarray(truefreq, dtype=np.float64), EPS, None)

    loss = 2.0 * expo * (
        lambda_pred
        - truefreq
        - truefreq * np.log(lambda_pred / truefreq)
    )

    return float(np.mean(loss))


def poisson_nll_from_log_mu_torch(y, log_mu):
    """
    Poisson negative log-likelihood without constant log(y!).

    loss = mean(exp(log_mu) - y * log_mu)
    """
    mu = torch.exp(log_mu)
    return torch.mean(mu - y * log_mu)


# ============================================================
# 5. GLM baseline
# ============================================================

def fit_glm(train_df: pd.DataFrame):
    """
    Poisson GLM with log link and offset log(expo).
    """
    formula = (
        "claims ~ C(powerGLM) + area_num + log_dens + C(gas) "
        "+ C(ageGLM) + C(acGLM) + C(brand) + C(ct)"
    )

    model = smf.glm(
        formula=formula,
        data=train_df,
        family=sm.families.Poisson(),
        offset=np.log(np.clip(train_df["expo"].values, EPS, None)),
    )

    result = model.fit()
    return result


def predict_glm_lambda(glm_result, df: pd.DataFrame) -> np.ndarray:
    """
    Return predicted frequency lambda_hat, not predicted count mu_hat.
    """
    expo = np.clip(df["expo"].values.astype(np.float64), EPS, None)

    mu_hat = glm_result.predict(
        df,
        offset=np.log(expo),
    )

    lambda_hat = np.asarray(mu_hat, dtype=np.float64) / expo
    lambda_hat = np.clip(lambda_hat, EPS, None)

    return lambda_hat


def evaluate_lambda_predictions(model_name, split_name, df, lambda_pred, seed=None):
    expo = df["expo"].values.astype(np.float64)
    y = df["claims"].values.astype(np.float64)

    mu_pred = expo * lambda_pred

    return {
        "model": model_name,
        "seed": seed,
        "split": split_name,
        "poisson_deviance": poisson_deviance(y, mu_pred),
        "estimation_loss_truefreq": estimation_loss_truefreq(
            expo=expo,
            lambda_pred=lambda_pred,
            truefreq=df["truefreq"].values.astype(np.float64),
        ),
    }


# ============================================================
# 6. PyTorch dataset
# ============================================================

class MTPLDataset(Dataset):
    def __init__(self, X, claims, expo, glm_lambda=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(claims.astype(np.float32), dtype=torch.float32)
        self.expo = torch.tensor(expo.astype(np.float32), dtype=torch.float32)

        log_expo = np.log(np.clip(expo.astype(np.float64), EPS, None))
        self.log_expo = torch.tensor(log_expo, dtype=torch.float32)

        if glm_lambda is None:
            glm_lambda = np.ones_like(expo, dtype=np.float64)

        log_glm_lambda = np.log(np.clip(glm_lambda.astype(np.float64), EPS, None))
        self.log_glm_lambda = torch.tensor(log_glm_lambda, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x": self.X[idx],
            "y": self.y[idx],
            "expo": self.expo[idx],
            "log_expo": self.log_expo[idx],
            "log_glm_lambda": self.log_glm_lambda[idx],
        }


# ============================================================
# 7. Neural models
# ============================================================

class DNNFrequencyModel(nn.Module):
    """
    Ordinary DNN:

    log_mu = log_expo + f_theta(x)
    """
    def __init__(self, input_dim: int):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x, log_expo, log_glm_lambda=None):
        log_lambda = self.net(x).squeeze(-1)
        return log_expo + log_lambda


class CANNFrequencyModel(nn.Module):
    """
    CANN:

    log_mu = log_expo + log_glm_lambda + g_theta(x)

    Final layer initialized to zero, so initially:

    g_theta(x) = 0

    Therefore initial CANN prediction equals GLM prediction.
    """
    def __init__(self, input_dim: int):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

        final_layer = self.net[-1]
        nn.init.zeros_(final_layer.weight)
        nn.init.zeros_(final_layer.bias)

    def forward(self, x, log_expo, log_glm_lambda):
        residual = self.net(x).squeeze(-1)
        return log_expo + log_glm_lambda + residual


# ============================================================
# 8. Training and prediction
# ============================================================

def train_model(model, train_loader, val_loader, device):
    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_loss = np.inf
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []

        for batch in train_loader:
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            log_expo = batch["log_expo"].to(device)
            log_glm_lambda = batch["log_glm_lambda"].to(device)

            optimizer.zero_grad()

            log_mu = model(
                x=x,
                log_expo=log_expo,
                log_glm_lambda=log_glm_lambda,
            )

            loss = poisson_nll_from_log_mu_torch(y, log_mu)

            if torch.isnan(loss) or torch.isinf(loss):
                raise RuntimeError("Training loss became NaN or Inf.")

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))

        model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                x = batch["x"].to(device)
                y = batch["y"].to(device)
                log_expo = batch["log_expo"].to(device)
                log_glm_lambda = batch["log_glm_lambda"].to(device)

                log_mu = model(
                    x=x,
                    log_expo=log_expo,
                    log_glm_lambda=log_glm_lambda,
                )

                loss = poisson_nll_from_log_mu_torch(y, log_mu)
                val_losses.append(loss.item())

        val_loss = float(np.mean(val_losses))

        print(
            f"Epoch {epoch:03d} | "
            f"train NLL: {train_loss:.6f} | "
            f"val NLL: {val_loss:.6f}"
        )

        if val_loss < best_val_loss - 1e-8:
            best_val_loss = val_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def predict_nn_lambda(model, data_loader, device):
    model.eval()
    model.to(device)

    all_mu = []
    all_expo = []

    with torch.no_grad():
        for batch in data_loader:
            x = batch["x"].to(device)
            log_expo = batch["log_expo"].to(device)
            log_glm_lambda = batch["log_glm_lambda"].to(device)

            log_mu = model(
                x=x,
                log_expo=log_expo,
                log_glm_lambda=log_glm_lambda,
            )

            log_mu = torch.clamp(log_mu, min=-30.0, max=20.0)

            mu = torch.exp(log_mu).detach().cpu().numpy()
            expo = batch["expo"].detach().cpu().numpy()

            all_mu.append(mu)
            all_expo.append(expo)

    mu_pred = np.concatenate(all_mu)
    expo = np.concatenate(all_expo)

    lambda_pred = mu_pred / np.clip(expo, EPS, None)
    lambda_pred = np.clip(lambda_pred, EPS, None)

    return lambda_pred


def make_loaders(
    X_train,
    X_val,
    X_test,
    train_df,
    val_df,
    test_df,
    glm_train=None,
    glm_val=None,
    glm_test=None,
):
    train_dataset = MTPLDataset(
        X=X_train,
        claims=train_df["claims"].values,
        expo=train_df["expo"].values,
        glm_lambda=glm_train,
    )

    val_dataset = MTPLDataset(
        X=X_val,
        claims=val_df["claims"].values,
        expo=val_df["expo"].values,
        glm_lambda=glm_val,
    )

    test_dataset = MTPLDataset(
        X=X_test,
        claims=test_df["claims"].values,
        expo=test_df["expo"].values,
        glm_lambda=glm_test,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    train_eval_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, train_eval_loader, val_loader, test_loader


# ============================================================
# 9. Main experiment
# ============================================================

def main():
    warnings.filterwarnings("ignore")
    ensure_results_dir()

    print("=" * 80)
    print("GLM vs DNN vs CANN experiment")
    print("=" * 80)
    print(f"Using device: {DEVICE}")
    print(f"Data path: {DATA_PATH}")
    print(f"Smoke test: {SMOKE_TEST}")

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------
    print("\nLoading data...")
    df = load_raw_data(DATA_PATH)

    if SMOKE_TEST:
        df = df.head(SMOKE_N_ROWS).copy()
        print(f"Smoke test enabled. Using first {SMOKE_N_ROWS} rows.")

    df = add_model_features(df)

    print(f"Data shape: {df.shape}")
    print("Columns:", list(df.columns))
    print(df.head())

    print("\nClaim distribution:")
    print(df["claims"].value_counts().head(10))
    print(f"Claim frequency in data: {df['claims'].sum() / df['expo'].sum():.6f}")

    # --------------------------------------------------------
    # Split data
    # --------------------------------------------------------
    train_df, val_df, test_df = make_train_val_test_split(df)

    print("\nData split:")
    print(f"Train shape: {train_df.shape}")
    print(f"Val shape:   {val_df.shape}")
    print(f"Test shape:  {test_df.shape}")

    # --------------------------------------------------------
    # NN preprocessing
    # --------------------------------------------------------
    print("\nBuilding neural network features...")
    preprocessor = build_nn_preprocessor(train_df)

    X_train = transform_nn_features(preprocessor, train_df)
    X_val = transform_nn_features(preprocessor, val_df)
    X_test = transform_nn_features(preprocessor, test_df)

    input_dim = X_train.shape[1]
    print(f"Neural network input dimension: {input_dim}")

    # --------------------------------------------------------
    # GLM
    # --------------------------------------------------------
    print("\nFitting Poisson GLM...")
    glm_result = fit_glm(train_df)

    print("\nGLM fitted.")
    print(glm_result.summary())

    glm_lambda_train = predict_glm_lambda(glm_result, train_df)
    glm_lambda_val = predict_glm_lambda(glm_result, val_df)
    glm_lambda_test = predict_glm_lambda(glm_result, test_df)

    all_metrics = []

    for split_name, split_df, glm_lambda in [
        ("train", train_df, glm_lambda_train),
        ("val", val_df, glm_lambda_val),
        ("test", test_df, glm_lambda_test),
    ]:
        all_metrics.append(
            evaluate_lambda_predictions(
                model_name="GLM",
                split_name=split_name,
                df=split_df,
                lambda_pred=glm_lambda,
                seed=None,
            )
        )

    predictions_test = pd.DataFrame({
        "id": test_df["id"].values,
        "claims": test_df["claims"].values,
        "expo": test_df["expo"].values,
        "truefreq": test_df["truefreq"].values,
        "glm_lambda": glm_lambda_test,
    })

    # --------------------------------------------------------
    # DNN and CANN
    # --------------------------------------------------------
    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(f"Seed {seed}")
        print("=" * 80)

        set_seed(seed)

        # ------------------------------
        # Ordinary DNN
        # ------------------------------
        print("\nTraining ordinary DNN...")

        dnn_train_loader, dnn_train_eval_loader, dnn_val_loader, dnn_test_loader = make_loaders(
            X_train=X_train,
            X_val=X_val,
            X_test=X_test,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            glm_train=None,
            glm_val=None,
            glm_test=None,
        )

        dnn_model = DNNFrequencyModel(input_dim=input_dim)

        dnn_model = train_model(
            model=dnn_model,
            train_loader=dnn_train_loader,
            val_loader=dnn_val_loader,
            device=DEVICE,
        )

        dnn_lambda_train = predict_nn_lambda(dnn_model, dnn_train_eval_loader, DEVICE)
        dnn_lambda_val = predict_nn_lambda(dnn_model, dnn_val_loader, DEVICE)
        dnn_lambda_test = predict_nn_lambda(dnn_model, dnn_test_loader, DEVICE)

        for split_name, split_df, lambda_pred in [
            ("train", train_df, dnn_lambda_train),
            ("val", val_df, dnn_lambda_val),
            ("test", test_df, dnn_lambda_test),
        ]:
            all_metrics.append(
                evaluate_lambda_predictions(
                    model_name="DNN",
                    split_name=split_name,
                    df=split_df,
                    lambda_pred=lambda_pred,
                    seed=seed,
                )
            )

        predictions_test[f"dnn_lambda_seed{seed}"] = dnn_lambda_test

        # ------------------------------
        # CANN
        # ------------------------------
        print("\nTraining CANN...")

        cann_train_loader, cann_train_eval_loader, cann_val_loader, cann_test_loader = make_loaders(
            X_train=X_train,
            X_val=X_val,
            X_test=X_test,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            glm_train=glm_lambda_train,
            glm_val=glm_lambda_val,
            glm_test=glm_lambda_test,
        )

        cann_model = CANNFrequencyModel(input_dim=input_dim)

        cann_model = train_model(
            model=cann_model,
            train_loader=cann_train_loader,
            val_loader=cann_val_loader,
            device=DEVICE,
        )

        cann_lambda_train = predict_nn_lambda(cann_model, cann_train_eval_loader, DEVICE)
        cann_lambda_val = predict_nn_lambda(cann_model, cann_val_loader, DEVICE)
        cann_lambda_test = predict_nn_lambda(cann_model, cann_test_loader, DEVICE)

        for split_name, split_df, lambda_pred in [
            ("train", train_df, cann_lambda_train),
            ("val", val_df, cann_lambda_val),
            ("test", test_df, cann_lambda_test),
        ]:
            all_metrics.append(
                evaluate_lambda_predictions(
                    model_name="CANN",
                    split_name=split_name,
                    df=split_df,
                    lambda_pred=lambda_pred,
                    seed=seed,
                )
            )

        predictions_test[f"cann_lambda_seed{seed}"] = cann_lambda_test

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------
    metrics_df = pd.DataFrame(all_metrics)

    metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")
    predictions_path = os.path.join(RESULTS_DIR, "predictions_test.csv")

    metrics_df.to_csv(metrics_path, index=False)
    predictions_test.to_csv(predictions_path, index=False)

    print("\n" + "=" * 80)
    print("Experiment completed.")
    print(f"Metrics saved to: {metrics_path}")
    print(f"Test predictions saved to: {predictions_path}")
    print("=" * 80)

    print("\nAll metrics:")
    print(metrics_df)

    print("\nTest-set comparison:")
    test_metrics = metrics_df[metrics_df["split"] == "test"].copy()
    print(test_metrics.sort_values(["model", "seed"]))

    print("\nMean test-set results by model:")
    print(
        test_metrics
        .groupby("model")[["poisson_deviance", "estimation_loss_truefreq"]]
        .agg(["mean", "std"])
    )


if __name__ == "__main__":
    main()

GLM vs DNN vs CANN experiment
Using device: cpu
Data path: C:\Users\Guorui\Desktop\4TH ULM\SEMINAR-INSURANCE\MTPL_data_clean.csv
Smoke test: True

Loading data...
Smoke test enabled. Using first 10000 rows.
Data shape: (10000, 17)
Columns: ['id', 'claims', 'expo', 'age', 'ac', 'power', 'gas', 'brand', 'area', 'dens', 'ct', 'truefreq', 'log_dens', 'area_num', 'ageGLM', 'acGLM', 'powerGLM']
   id  claims  expo  age  ac  power      gas brand area  dens  ct  truefreq  \
0   1       0  0.33   66   4      3  Regular   B12    B    83  BE  0.059941   
1   2       0  0.08   31   1      7   Diesel    B1    A    34  BL  0.119216   
2   3       0  0.92   60   6      5   Diesel    B1    C   223  AG  0.074344   
3   4       0  1.00   66   4      2   Diesel    B1    C   283  FR  0.092829   
4   5       0  0.63   63   3      5  Regular   B12    B    74  VS  0.049981   

   log_dens  area_num  ageGLM    acGLM powerGLM  
0  4.418841         2  61to70  ac3plus        3  
1  3.526361         1  31to40    